# Arm A — Global (Non-Stratified) Model & Evaluation

Trains a single classifier on the full Cleveland cohort (no patient
stratification) to serve as the common baseline for RQ1 (vs. clinically
defined subgroups, Arm B) and RQ2 (vs. data-driven clusters, Arm C). Per
proposal §3.3–3.5:

- Two classifiers, same procedure: logistic regression (interpretable
  linear) and random forest (non-linear comparator).
- Preprocessing (encoding, scaling) is fit on training folds only, to
  avoid leakage.
- Evaluated with stratified k-fold CV; the fold assignment is fixed and
  saved here so Arms B and C can reuse the *same* patient-to-fold mapping
  for a matched comparison.
- Metrics: accuracy, F1-score, ROC-AUC, reported as mean ± std across
  folds.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# restecg == 1 has only 4/297 patients, so some CV training folds miss it
# entirely; OneHotEncoder(handle_unknown="ignore") correctly falls back to
# an all-zero encoding for it in the held-out fold, which is expected here
# and not a bug — this just silences the resulting per-fold warning.
warnings.filterwarnings("ignore", message="Found unknown categories", category=UserWarning)

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

RESULTS_DIR = "results"
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

## Load the cleaned dataset

Reads `heart+disease/cleveland_clean.csv` (produced by `data_cleaning.ipynb`).
The CSV round-trip loses pandas' `category` dtype, so the nominal columns
are re-cast here. `num` (0–4 severity) is dropped from the feature matrix —
it's what `check` was derived from, so keeping it would leak the label.

In [2]:
COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "num", "check",
    ]
NOMINAL_FEATURES = ["cp", "restecg", "slope", "thal"]
BINARY_FEATURES = ["sex", "fbs", "exang"]
NUMERIC_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
FEATURE_COLUMNS = NOMINAL_FEATURES + BINARY_FEATURES + NUMERIC_FEATURES

df = pd.read_csv("heart+disease/cleveland_clean.csv")
assert list(df.columns) == COLUMN_NAMES, "unexpected column layout in cleveland_clean.csv"

for col in NOMINAL_FEATURES:
    df[col] = df[col].astype("category")

X = df[FEATURE_COLUMNS].copy()
y = df["check"].astype(int)

print(df.shape)
print(y.value_counts())

(297, 15)
check
0    160
1    137
Name: count, dtype: int64


## Shared preprocessing pipeline

`ColumnTransformer` one-hot encodes the nominal clinical codes, standardises
the continuous/count features, and passes the already-binary 0/1 features
through unchanged. This is wrapped in a `Pipeline` with the classifier so
that `fit` (including the scaler's mean/std) only ever sees the training
fold — the encoder/scaler are refit inside every CV split, never on the
full dataset.

In [3]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("nominal", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), NOMINAL_FEATURES),
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
            ("binary", "passthrough", BINARY_FEATURES),
            ]
        )

## Fix the cross-validation split

A single `StratifiedKFold` split, seeded once, is used for every arm in
this study. Saving the per-patient fold assignment now (indexed to
`cleveland_clean.csv` row order) lets Arm B and Arm C evaluate their
subgroup/cluster models against identical held-out folds, so differences
in results reflect the stratification strategy rather than a different
train/test split.

In [4]:
outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_assignment = np.full(len(df), -1, dtype=int)
for fold_id, (_, test_idx) in enumerate(outer_cv.split(X, y)):
    fold_assignment[test_idx] = fold_id

fold_df = pd.DataFrame({"row_index": np.arange(len(df)), "fold": fold_assignment})
fold_df.to_csv(f"{RESULTS_DIR}/cv_fold_assignment.csv", index=False)

print(pd.Series(fold_assignment).value_counts().sort_index())

0    60
1    60
2    59
3    59
4    59
Name: count, dtype: int64


## Classifiers and hyperparameter grids

Both classifiers use the same nested-CV selection procedure: for each
outer training fold, `GridSearchCV` picks hyperparameters via an inner
3-fold split, scoring on ROC-AUC. Grids are deliberately small — the study
compares stratification strategies, not classifier tuning.

In [5]:
MODEL_SPECS = {
    "logistic_regression": {
        "estimator": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "param_grid": {"clf__C": [0.01, 0.1, 1, 10, 100]},
        },
    "random_forest": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "clf__n_estimators": [100, 300],
            "clf__max_depth": [None, 3, 5],
            "clf__min_samples_leaf": [1, 2, 5],
            },
        },
    }

## Run nested CV

For every classifier and every outer fold: fit `GridSearchCV` on the
training fold (inner CV picks hyperparameters), then score the held-out
outer fold. Out-of-fold predictions are kept so all 297 patients end up
with exactly one prediction each, directly comparable to the pooled
subgroup/cluster predictions in Arms B and C.

In [6]:
def run_nested_cv(name, spec):
    pipeline = Pipeline([("prep", make_preprocessor()), ("clf", spec["estimator"])])
    inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    fold_metrics = []
    oof_pred = np.full(len(df), np.nan)
    oof_proba = np.full(len(df), np.nan)
    best_params_per_fold = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        search = GridSearchCV(pipeline, spec["param_grid"], cv=inner_cv, scoring="roc_auc", n_jobs=-1)
        search.fit(X_train, y_train)

        pred = search.predict(X_test)
        proba = search.predict_proba(X_test)[:, 1]

        oof_pred[test_idx] = pred
        oof_proba[test_idx] = proba
        best_params_per_fold.append(search.best_params_)

        fold_metrics.append({
            "model": name,
            "fold": fold_id,
            "accuracy": accuracy_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "best_params": search.best_params_,
            })

    return fold_metrics, oof_pred, oof_proba, best_params_per_fold


all_fold_metrics = []
oof_predictions = {"row_index": np.arange(len(df)), "y_true": y.values}

for name, spec in MODEL_SPECS.items():
    fold_metrics, oof_pred, oof_proba, best_params = run_nested_cv(name, spec)
    all_fold_metrics.extend(fold_metrics)
    oof_predictions[f"{name}_pred"] = oof_pred
    oof_predictions[f"{name}_proba"] = oof_proba
    print(f"{name}: best params per fold = {best_params}")

fold_metrics_df = pd.DataFrame(all_fold_metrics)
fold_metrics_df

logistic_regression: best params per fold = [{'clf__C': 0.1}, {'clf__C': 1}, {'clf__C': 1}, {'clf__C': 0.1}, {'clf__C': 0.1}]


random_forest: best params per fold = [{'clf__max_depth': 5, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 100}, {'clf__max_depth': 3, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 300}, {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 300}, {'clf__max_depth': 3, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 300}, {'clf__max_depth': 3, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 100}]


,model,fold,accuracy,f1,roc_auc,best_params
0,logistic_regression,0,0.916667,0.909091,0.947545,{'clf__C': 0.1}
1,logistic_regression,1,0.833333,0.814815,0.908482,{'clf__C': 1}
2,logistic_regression,2,0.745763,0.727273,0.834491,{'clf__C': 1}
3,logistic_regression,3,0.847458,0.816327,0.888889,{'clf__C': 0.1}
4,logistic_regression,4,0.864407,0.833333,0.936343,{'clf__C': 0.1}
5,random_forest,0,0.883333,0.877193,0.946429,"{'clf__max_depth': 5, 'clf__min_samples_leaf':..."
6,random_forest,1,0.816667,0.800000,0.919643,"{'clf__max_depth': 3, 'clf__min_samples_leaf':..."
7,random_forest,2,0.694915,0.666667,0.825231,"{'clf__max_depth': None, 'clf__min_samples_lea..."
8,random_forest,3,0.847458,0.823529,0.894676,"{'clf__max_depth': 3, 'clf__min_samples_leaf':..."
9,random_forest,4,0.898305,0.875000,0.961806,"{'clf__max_depth': 3, 'clf__min_samples_leaf':..."


## Aggregate results

Mean ± std across the 5 outer folds, per classifier — this table is the
Arm A baseline that Arm B and Arm C results will be compared against.

In [7]:
summary = (
    fold_metrics_df
    .groupby("model")[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
    )
summary

accuracy                  f1             roc_auc  \
                         mean       std      mean       std      mean   
model                                                                   
logistic_regression  0.841525  0.062134  0.820168  0.064718  0.903150   
random_forest        0.828136  0.080968  0.808478  0.085959  0.909557   

                               
                          std  
model                          
logistic_regression  0.044773  
random_forest        0.053674

## Persist results

Saves the per-fold metrics, the summary table, and out-of-fold predictions
for later cross-arm comparison, plus (already saved above) the fold
assignment that Arms B and C must reuse.

In [8]:
fold_metrics_df.to_csv(f"{RESULTS_DIR}/arm_a_fold_metrics.csv", index=False)
summary.to_csv(f"{RESULTS_DIR}/arm_a_summary.csv")
pd.DataFrame(oof_predictions).to_csv(f"{RESULTS_DIR}/arm_a_oof_predictions.csv", index=False)

print("Saved:")
print(f"  {RESULTS_DIR}/cv_fold_assignment.csv")
print(f"  {RESULTS_DIR}/arm_a_fold_metrics.csv")
print(f"  {RESULTS_DIR}/arm_a_summary.csv")
print(f"  {RESULTS_DIR}/arm_a_oof_predictions.csv")

Saved:
  results/cv_fold_assignment.csv
  results/arm_a_fold_metrics.csv
  results/arm_a_summary.csv
  results/arm_a_oof_predictions.csv


## Interpretability snapshot (context only)

Refits each classifier on the *full* dataset (not for evaluation — purely
to inspect what the global model relies on, for later discussion when
comparing against subgroup/cluster models in Arms B and C).

In [9]:
full_pipelines = {}
for name, spec in MODEL_SPECS.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", spec["estimator"])])
    pipe.fit(X, y)
    full_pipelines[name] = pipe

feature_names = full_pipelines["logistic_regression"].named_steps["prep"].get_feature_names_out()

lr_coefs = pd.Series(
    full_pipelines["logistic_regression"].named_steps["clf"].coef_[0], index=feature_names,
    ).sort_values(key=np.abs, ascending=False)
print("Logistic regression coefficients (full-data refit):")
print(lr_coefs)

rf_importances = pd.Series(
    full_pipelines["random_forest"].named_steps["clf"].feature_importances_, index=feature_names,
    ).sort_values(ascending=False)
print()
print("Random forest feature importances (full-data refit):")
print(rf_importances)

Logistic regression coefficients (full-data refit):
binary__sex             1.149453
numeric__ca             1.087771
nominal__cp_4.0         1.042068
nominal__thal_7.0       0.822530
nominal__cp_1.0        -0.633481
binary__exang           0.625859
nominal__cp_3.0        -0.580127
nominal__thal_3.0      -0.544446
nominal__slope_2.0      0.493892
nominal__slope_1.0     -0.438481
binary__fbs            -0.423662
numeric__oldpeak        0.411396
numeric__thalach       -0.394021
numeric__trestbps       0.363585
nominal__thal_6.0      -0.275997
nominal__restecg_0.0   -0.264553
numeric__chol           0.187847
nominal__restecg_2.0    0.184611
nominal__cp_2.0         0.173626
numeric__age           -0.106943
nominal__restecg_1.0    0.082028
nominal__slope_3.0     -0.053324
dtype: float64

Random forest feature importances (full-data refit):
numeric__ca             0.117385
numeric__thalach        0.109618
numeric__oldpeak        0.092454
numeric__age            0.086977
nominal__thal_3.0    